# 02 - Training comparison (4 targets x 6 arms)

> STATUS: pipeline smoke-tested (CAD, 2 folds x 1 repeat). FULL 4-target run
> (5 folds x 3 repeats, `uv run python -m app.train`, ~15-40 min CPU) is PENDING.
> Re-execute this notebook after the full run; every cell below reads
> `backend/artifacts/cv_results.csv` and adapts to whatever targets exist.
>
> Selection rule (no winner picked here): best mean ROC-AUC per target,
> tie-break on F1/recall. Unbiased estimates come from the Phase 5 holdout.

Run from `backend/` with:
`PYTHONPATH=$PWD uv run jupyter nbconvert --to notebook --execute ../notebooks/02_training.ipynb`

In [ ]:
import pandas as pd
from app.config import ARTIFACTS_DIR  # absolute path, independent of kernel cwd

res = pd.read_csv(ARTIFACTS_DIR / 'cv_results.csv')
print('rows:', len(res), '| targets present:', sorted(res['target'].unique().tolist()))
missing = {'CAD', 'LAD', 'LCX', 'RCA'} - set(res['target'].unique())
if missing:
    print('PENDING full-run targets:', sorted(missing))
cols = ['target', 'model', 'roc_auc_mean', 'roc_auc_std', 'f1_mean', 'f1_std',
        'recall_mean', 'accuracy_mean', 'precision_mean']
print(res[cols].sort_values(['target', 'roc_auc_mean'], ascending=[True, False]).to_string(index=False))

## ROC-AUC per model and target (mean +/- std across folds)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pivot = res.pivot(index='model', columns='target', values='roc_auc_mean')
err = res.pivot(index='model', columns='target', values='roc_auc_std')
ax = pivot.plot.bar(yerr=err, capsize=2, figsize=(9, 4), title='CV ROC-AUC by model and target')
ax.set_ylim(0.5, 1.0)
ax.figure.tight_layout()

## F1 vs recall (bubble size = ROC-AUC): screening trade-off per arm

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for _, r in res.iterrows():
    ax.scatter(r['recall_mean'], r['f1_mean'], s=60 + 400 * (r['roc_auc_mean'] - 0.5),
               alpha=0.7, label=f"{r['target']}/{r['model']}")
    ax.annotate(r['model'], (r['recall_mean'], r['f1_mean']), fontsize=7)
ax.set_xlabel('recall (mean)')
ax.set_ylabel('F1 (mean)')
ax.set_title('Screening trade-off: recall vs F1')
fig.tight_layout()